In [ ]:
import csv
from topologicpy.TGraph import TGraph


def _graphs_from_csv(graphs_path, edges_path, nodes_path):
    """Read the tutorial's three-file CSV layout into TGraph records."""
    def read(path):
        import ast
        with open(path, newline="", encoding="utf-8-sig") as stream:
            rows = list(csv.DictReader(stream))
        for row in rows:
            for key, value in row.items():
                if key in ("graph_id", "node_id", "src_id", "dst_id"):
                    continue
                try:
                    row[key] = ast.literal_eval(value)
                except (ValueError, SyntaxError):
                    pass
        return rows
    graphs, edges, nodes = read(graphs_path), read(edges_path), read(nodes_path)
    result = []
    for attributes in graphs:
        gid = attributes["graph_id"]
        selected = [v for v in nodes if v["graph_id"] == gid]
        indices = {v["node_id"]: i for i, v in enumerate(selected)}
        selected_edges = [e for e in edges if e["graph_id"] == gid]
        graph = TGraph.ByMeshData(
            [[float(v.get(k) or 0) for k in ("x", "y", "z")] for v in selected],
            [[indices[e["src_id"]], indices[e["dst_id"]]] for e in selected_edges],
            vertexDictionaries=selected, edgeDictionaries=selected_edges)
        graph.SetDictionary(attributes)
        result.append(graph)
    return {"graphs": result}


In [ ]:
# The following two lines are not needed if you have topologicpy already pip installed.
import sys
sys.path.append("C:/Users/wassimj/Documents/GitHub/")

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.TGraph import TGraph
from topologicpy.Plotly import Plotly
from topologicpy.Color import Color
from topologicpy.Vector import Vector
from topologicpy.Helper import Helper
from topologicpy.DGL import DGL
import math
print("Done importing libraries")

In [ ]:
# Load the training/validation graphs, edges, and nodes from CSV files
graphs_file_path = "C:/Users/wassimj/Downloads/acadia_csv_test04/test_graphs.csv"
edges_file_path = "C:/Users/wassimj/Downloads/acadia_csv_test04/test_edges.csv"
nodes_file_path = "C:/Users/wassimj/Downloads/acadia_csv_test04/test_nodes.csv"
brep_file_path = "C:/Users/wassimj/Downloads/acadia_csv_test04/test.brep"

# Import the BREP file. Should yield a CellComplex.
cc = Topology.ByBREPPath(brep_file_path)
print(cc)

# Import the dual graph from the Grasshopper CSV export.
dic01 = _graphs_from_csv(graphs_file_path, edges_file_path, nodes_file_path)
print(dic01)
graph = dic01['graphs'][0]
print(graph)

# Create Plotly Data for CellComplex and Graph
building_data= Plotly.DataByTopology(cc, faceOpacity=0.1)
graph_data = Plotly.DataByGraph(graph, vertexSize=3, vertexGroupKey="label", vertexGroups=[0,1,2,3,4])

# Create a Plotly Figure and Show it
fig = Plotly.FigureByData(graph_data+building_data)
Plotly.Show(fig)

# Create DGL Graphs from the CSV files. DGL Graphs are different from Topologic Graphs.
dic02 = DGL.GraphsByCSVPath(graphs_file_path, edges_file_path, nodes_file_path)
print(dic02)
# Create a DGL dataset directly from the CSV files.
dataset = DGL. DatasetByCSVPath(graphs_file_path, edges_file_path, nodes_file_path)
print(dataset)
# If this dataset contained many graphs, it would be ready for training a Graph Neural Network.